# Pipeline 4 - Diagnostico do SVM ATUALIZADA

Versao atualizada com split 75/10/15, distribuicao de classes, FNR, AUC/ROC, distribuicao de scores, busca ampliada de `C`, comparacao BoW e inspecao de pesos.


## 1. Carregar embeddings + rotulos + textos

A Pipeline 3 deve gerar embeddings usando **apenas titulo + abstract**, sem `id` e sem categoria/subgenero no texto de entrada. O `assigned_category` aparece aqui somente como rotulo supervisionado (`y`).

In [ ]:
from google.colab import drive
import os, numpy as np, pandas as pd

drive.mount('/content/drive')

# If needed, put the exact folder where Pipeline 3 saved the updated files.
# Example: BASE = '/content/drive/MyDrive/nome-da-pasta/pipelines'
BASE = None
NOME_EMB = 'arxiv_amostra_1500_embeddings_atualizada.npy'
NOME_JSONL = 'arxiv_amostra_1500_com_embeddings_atualizada.json'


def shallow_find(root, filename, max_depth=3):
    root = os.path.abspath(root)
    root_depth = root.rstrip(os.sep).count(os.sep)
    for current, dirs, files in os.walk(root):
        depth = current.rstrip(os.sep).count(os.sep) - root_depth
        if depth >= max_depth:
            dirs[:] = []
        if filename in files:
            return os.path.join(current, filename)
    return None


def encontrar_obrigatorio(nome):
    candidatos = []
    if BASE:
        candidatos.extend([
            os.path.join(BASE, nome),
            os.path.join(BASE, 'pipelines', nome),
        ])

    candidatos.extend([
        os.path.join('/content/drive/MyDrive', nome),
        os.path.join('/content/drive/MyDrive', 'pipelines', nome),
    ])

    encontrado = next((p for p in candidatos if os.path.exists(p)), None)
    if encontrado:
        return encontrado

    print(f'{nome} not found in direct paths. Starting shallow search in MyDrive, max_depth=3...')
    encontrado = shallow_find('/content/drive/MyDrive', nome, max_depth=3)
    if encontrado:
        return encontrado

    print(f'Required file not found: {nome}')
    print('\nDirect paths tested:')
    for p in candidatos:
        print(' -', p)
    print('\nTop-level items in /content/drive/MyDrive:')
    try:
        for item in sorted(os.listdir('/content/drive/MyDrive'))[:120]:
            print(' -', os.path.join('/content/drive/MyDrive', item))
    except Exception as exc:
        print('Could not list MyDrive:', repr(exc))
    raise FileNotFoundError(nome)

EMB = encontrar_obrigatorio(NOME_EMB)
JSONL = encontrar_obrigatorio(NOME_JSONL)
PIPE = os.path.dirname(JSONL)

X  = np.load(EMB)
df = pd.read_json(JSONL, lines=True)
assert len(df) == len(X), 'embeddings e textos com tamanhos diferentes!'

y = df['assigned_category'].to_numpy(dtype=object)

titulo = df['title'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
abstract = df['abstract_reduzido'].fillna(df.get('abstract', '')).astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

# Text inputs for baselines. Do not include id or assigned_category.
textos_title_abs = (titulo + '. ' + abstract).str.strip().to_numpy()
textos_abs_only = abstract.to_numpy()

ORDEM_CATS = ['cs.LG', 'cs.AI', 'cs.CL', 'cs.CV', 'hep-ph', 'hep-th', 'gr-qc',
              'quant-ph', 'astro-ph', 'math-ph', 'math.MP',
              'cond-mat.mtrl-sci', 'cond-mat.mes-hall', 'cond-mat.str-el', 'cond-mat.stat-mech']

print('Embedding file:', EMB)
print('JSONL file:', JSONL)
print('Output folder:', PIPE)
print('X:', X.shape, '| textos:', len(textos_title_abs), '| categorias:', len(set(y)))
print('Columns:', list(df.columns))
print('\nText input example without id/category:\n', textos_title_abs[0][:500])


## 2. Distribuicao de classes

Este grafico responde se o dataset esta balanceado e ajuda a explicar vies de classes. Se houver classes muito maiores, metricas macro e FNR por classe ficam mais importantes do que acuracia simples.

In [ ]:
import matplotlib.pyplot as plt

counts = pd.Series(y).value_counts().reindex(ORDEM_CATS).fillna(0).astype(int)
print(counts.to_string())
print('\nMaior classe:', counts.max(), '| Menor classe:', counts[counts > 0].min(), '| Razao max/min:', round(counts.max() / counts[counts > 0].min(), 2))

fig, ax = plt.subplots(figsize=(12, 4.8))
counts.plot(kind='bar', ax=ax, color='#2f6f9f')
ax.set_title('Distribuicao de classes - dataset completo')
ax.set_xlabel('Categoria')
ax.set_ylabel('Quantidade de artigos')
ax.tick_params(axis='x', rotation=65)
plt.tight_layout()

SAIDA_DIST = os.path.join(PIPE, 'distribuicao_classes_dataset_atualizada.png')
plt.savefig(SAIDA_DIST, dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em:', SAIDA_DIST)

## 3. Split estratificado: 75% treino / 10% validacao / 15% teste

O teste fica reservado para a avaliacao final. A validacao - usada para escolher hiperpar-metros (`C` e `class_weight`).

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(y))
idx_trainval, idx_te = train_test_split(idx, test_size=0.15, random_state=42, stratify=y)
idx_tr, idx_val = train_test_split(
    idx_trainval,
    test_size=0.10 / 0.85,
    random_state=42,
    stratify=y[idx_trainval],
)

X_tr, X_val, X_te = X[idx_tr], X[idx_val], X[idx_te]
y_tr, y_val, y_te = y[idx_tr], y[idx_val], y[idx_te]

txt_tr, txt_val, txt_te = textos_title_abs[idx_tr], textos_title_abs[idx_val], textos_title_abs[idx_te]
abs_tr, abs_val, abs_te = textos_abs_only[idx_tr], textos_abs_only[idx_val], textos_abs_only[idx_te]

print('Treino:', X_tr.shape, '| Validacao:', X_val.shape, '| Teste:', X_te.shape)
print('Proporcoes:', len(idx_tr)/len(idx), len(idx_val)/len(idx), len(idx_te)/len(idx))

split_counts = pd.DataFrame({
    'dataset': pd.Series(y).value_counts(),
    'treino': pd.Series(y_tr).value_counts(),
    'validacao': pd.Series(y_val).value_counts(),
    'teste': pd.Series(y_te).value_counts(),
}).reindex(ORDEM_CATS).fillna(0).astype(int)

ax = split_counts[['treino', 'validacao', 'teste']].plot(kind='bar', figsize=(13, 5))
ax.set_title('Distribuicao de classes por split')
ax.set_xlabel('Categoria')
ax.set_ylabel('Quantidade de artigos')
ax.tick_params(axis='x', rotation=65)
plt.tight_layout()

SAIDA_SPLITS = os.path.join(PIPE, 'distribuicao_classes_splits_atualizada.png')
plt.savefig(SAIDA_SPLITS, dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em:', SAIDA_SPLITS)

## 4. Funcoes de avaliacao

Alem de F1, adicionamos **FNR** (`FN / (FN + TP)`) porque ele mostra quanto o modelo deixa escapar de cada classe. Recall alto implica FNR baixo; quando recall parece alto demais, vale comparar com precisao e distribuicao dos scores.

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

C_GRID = [0.03, 0.1, 0.3, 1, 3, 10, 30, 100]
CLASS_WEIGHT_GRID = [None, 'balanced']


def fnr_por_classe(y_true, y_pred, labels):
    cm_local = confusion_matrix(y_true, y_pred, labels=labels)
    rows = []
    for i, label in enumerate(labels):
        tp = cm_local[i, i]
        fn = cm_local[i, :].sum() - tp
        fp = cm_local[:, i].sum() - tp
        tn = cm_local.sum() - tp - fn - fp
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        fnr = fn / (fn + tp) if (fn + tp) else 0.0
        rows.append({'classe': label, 'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn, 'recall': recall, 'FNR': fnr})
    return pd.DataFrame(rows)


def metricas_resumo(nome, y_true, y_pred):
    fnr_df = fnr_por_classe(y_true, y_pred, ORDEM_CATS)
    return {
        'modelo': nome,
        'acuracia': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'f1_micro': f1_score(y_true, y_pred, average='micro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'fnr_macro': fnr_df['FNR'].mean(),
    }


def imprimir_avaliacao(nome, y_true, y_pred):
    resumo = metricas_resumo(nome, y_true, y_pred)
    print(pd.Series(resumo).to_string())
    print('\nRelatorio por categoria:\n')
    print(classification_report(y_true, y_pred, labels=ORDEM_CATS, zero_division=0))
    print('\nFNR por classe:\n')
    print(fnr_por_classe(y_true, y_pred, ORDEM_CATS).round(3).to_string(index=False))
    return resumo

## 5. Treinar SVM sobre embeddings SPECTER

Busca automatizada de `C` e `class_weight`. Valores maiores de `C` foram incluidos porque `C=0.1` pode regularizar demais e causar underfitting.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

experimentos_specter = []
melhor = None

for C in C_GRID:
    for class_weight in CLASS_WEIGHT_GRID:
        clf_tmp = make_pipeline(
            StandardScaler(),
            SVC(kernel='linear', C=C, class_weight=class_weight, decision_function_shape='ovr'),
        )
        clf_tmp.fit(X_tr, y_tr)
        pred_tr = clf_tmp.predict(X_tr)
        pred_val = clf_tmp.predict(X_val)
        row = {
            'C': C,
            'class_weight': str(class_weight),
            'f1_macro_treino': f1_score(y_tr, pred_tr, average='macro'),
            'f1_macro_validacao': f1_score(y_val, pred_val, average='macro'),
            'balanced_acc_validacao': balanced_accuracy_score(y_val, pred_val),
        }
        experimentos_specter.append(row)
        if melhor is None or row['f1_macro_validacao'] > melhor[0]['f1_macro_validacao']:
            melhor = (row, clf_tmp)

resultados_c = pd.DataFrame(experimentos_specter).sort_values('f1_macro_validacao', ascending=False)
print(resultados_c.round(4).to_string(index=False))

best_row, clf = melhor
print('\nMelhor configuracao SPECTER:', best_row)

SAIDA_C_GRID = os.path.join(PIPE, 'busca_c_specter_atualizada.csv')
resultados_c.to_csv(SAIDA_C_GRID, index=False)
print('Tabela salva em:', SAIDA_C_GRID)

## 6. Avaliar SPECTER no teste final

In [ ]:
y_pred_val = clf.predict(X_val)
y_pred = clf.predict(X_te)

print('### Validacao')
res_specter_val = imprimir_avaliacao('SPECTER_validacao', y_val, y_pred_val)

print('\n\n### Teste final')
res_specter = imprimir_avaliacao('SPECTER_teste', y_te, y_pred)

## 7. Matriz de confusao

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_te, y_pred, labels=ORDEM_CATS)
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(cm, display_labels=ORDEM_CATS).plot(
    ax=ax, cmap='Blues', colorbar=False, xticks_rotation=90)
ax.set_title('Matriz de confusao - SVM sobre embeddings SPECTER')
plt.tight_layout()

SAIDA_FIG = os.path.join(PIPE, 'matriz_confusao_svm_atualizada.png')
plt.savefig(SAIDA_FIG, dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em:', SAIDA_FIG)

## 8. AUC/ROC e distribuicao dos scores

Para multiclasses, usamos one-vs-rest. O grafico de distribuicao mostra, para uma classe, se exemplos positivos ficam perto do extremo 1 e negativos perto do extremo 0. Quando ha muita massa no meio, o modelo esta inseguro; quando positivos e negativos se sobrepoem, ha confusao entre classes.

In [ ]:
scores = clf.decision_function(X_te)
classes_modelo = list(clf.named_steps['svc'].classes_)

Y_bin = label_binarize(y_te, classes=classes_modelo)
auc_macro = roc_auc_score(Y_bin, scores, average='macro', multi_class='ovr')
auc_weighted = roc_auc_score(Y_bin, scores, average='weighted', multi_class='ovr')
print('AUC macro OvR:   {:.3f}'.format(auc_macro))
print('AUC weighted OvR:{:.3f}'.format(auc_weighted))

# ROC one-vs-rest for all classes. The plot is dense, but useful as a global diagnostic.
fig, ax = plt.subplots(figsize=(9, 7))
auc_por_classe = []
for classe in ORDEM_CATS:
    if classe not in classes_modelo:
        continue
    j = classes_modelo.index(classe)
    fpr, tpr, _ = roc_curve(Y_bin[:, j], scores[:, j])
    auc_classe = auc(fpr, tpr)
    auc_por_classe.append({'classe': classe, 'auc_ovr': auc_classe})
    ax.plot(fpr, tpr, linewidth=1.2, label=f'{classe}={auc_classe:.2f}')
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax.set_title('ROC one-vs-rest - all classes')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR / Recall')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()

SAIDA_ROC = os.path.join(PIPE, 'roc_ovr_specter_atualizada.png')
plt.savefig(SAIDA_ROC, dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em:', SAIDA_ROC)

auc_df = pd.DataFrame(auc_por_classe).sort_values('auc_ovr')
print('\nAUC por classe, do pior para o melhor:')
print(auc_df.round(3).to_string(index=False))
SAIDA_AUC = os.path.join(PIPE, 'auc_por_classe_atualizada.csv')
auc_df.to_csv(SAIDA_AUC, index=False)
print('Tabela salva em:', SAIDA_AUC)

# Score distribution for all 15 classes.
# Blue = real class 0; orange = real class 1. Good separation means orange concentrated near 1
# and blue concentrated near 0, with little overlap in the middle/right side.
fig, axes = plt.subplots(5, 3, figsize=(16, 18), sharey=False)
axes = axes.ravel()

score_summary = []
for ax, classe in zip(axes, ORDEM_CATS):
    if classe not in classes_modelo:
        ax.axis('off')
        continue
    j = classes_modelo.index(classe)
    s = scores[:, j]
    s01 = (s - s.min()) / (s.max() - s.min() + 1e-12)
    positivos = y_te == classe
    negativos = ~positivos

    ax.hist(s01[negativos], bins=14, alpha=0.60, label='real=0', color='#4c78a8')
    ax.hist(s01[positivos], bins=14, alpha=0.80, label='real=1', color='#f58518')
    ax.set_title(classe)
    ax.set_xlim(0, 1)
    ax.grid(axis='y', alpha=0.18)

    if positivos.sum() > 0:
        score_summary.append({
            'classe': classe,
            'score_pos_medio': float(s01[positivos].mean()),
            'score_neg_medio': float(s01[negativos].mean()),
            'gap_medio_pos_neg': float(s01[positivos].mean() - s01[negativos].mean()),
            'pos_no_extremo_alto_0_8': float((s01[positivos] >= 0.8).mean()),
            'neg_no_extremo_alto_0_8': float((s01[negativos] >= 0.8).mean()),
        })

for ax in axes[len(ORDEM_CATS):]:
    ax.axis('off')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2)
fig.suptitle('Score distribution one-vs-rest - all classes', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])

SAIDA_SCORE_ALL = os.path.join(PIPE, 'distribuicao_scores_todas_classes_atualizada.png')
plt.savefig(SAIDA_SCORE_ALL, dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em:', SAIDA_SCORE_ALL)

score_summary_df = pd.DataFrame(score_summary).sort_values('gap_medio_pos_neg')
print('\nResumo da separacao dos scores, pior gap primeiro:')
print(score_summary_df.round(3).to_string(index=False))
SAIDA_SCORE_CSV = os.path.join(PIPE, 'resumo_distribuicao_scores_atualizada.csv')
score_summary_df.to_csv(SAIDA_SCORE_CSV, index=False)
print('Tabela salva em:', SAIDA_SCORE_CSV)


## 9. Onde estao os erros- Pares de categorias mais confundidos

In [ ]:
pares = []
for i in range(len(ORDEM_CATS)):
    for j in range(i + 1, len(ORDEM_CATS)):
        n = cm[i, j] + cm[j, i]
        if n > 0:
            pares.append((n, ORDEM_CATS[i], ORDEM_CATS[j], cm[i, j], cm[j, i]))
pares.sort(reverse=True)
print('Pares mais confundidos (total, classeA, classeB, A->B, B->A):')
for item in pares[:15]:
    print('{:3d} {:18s} - {:18s} | {} -> {} / {} -> {}'.format(item[0], item[1], item[2], item[1], item[2], item[2], item[1]))

## 10. Inspecao dos pesos do SVM linear

No SVM linear, os pesos indicam a direcao do hiperplano. Com `SVC(kernel='linear')`, o `coef_` - organizado em classificadores one-vs-one; em embeddings SPECTER os pesos sao dimensoes latentes, ent-o sao menos interpretaveis do que palavras do BoW, mas ajudam a medir norma/margem e classes mais proximas.

In [ ]:
svc = clf.named_steps['svc']
print('Classes:', svc.classes_)
print('coef_.shape:', svc.coef_.shape)
print('intercept_.shape:', svc.intercept_.shape)

normas = np.linalg.norm(svc.coef_, axis=1)
print('\nNormas dos hiperplanos one-vs-one:')
print(pd.Series(normas).describe().round(4).to_string())

# As maiores dimensoes latentes de alguns hiperplanos; util para auditoria, nao para interpretacao semantica direta.
for k in range(min(5, svc.coef_.shape[0])):
    top_dims = np.argsort(np.abs(svc.coef_[k]))[::-1][:10]
    print(f'\nHiperplano OVO #{k} | intercept={svc.intercept_[k]:.4f}')
    print('Top dimensoes SPECTER:', top_dims.tolist())
    print('Pesos:', np.round(svc.coef_[k, top_dims], 4).tolist())

## 11. Analise das categorias-irmas: `math-ph` e `math.MP`

In [ ]:
def fundir(arr):
    return np.array(['math-ph/MP' if v in ('math-ph', 'math.MP') else v for v in arr], dtype=object)

y_te_f, y_pred_f = fundir(y_te), fundir(y_pred)
print('Acuracia original: {:.3f}'.format(accuracy_score(y_te, y_pred)))
print('Acuracia fundindo math-ph + math.MP: {:.3f}'.format(accuracy_score(y_te_f, y_pred_f)))
print('F1-macro fundindo math-ph + math.MP: {:.3f}'.format(f1_score(y_te_f, y_pred_f, average='macro')))

## 12. Baseline Bag of Words e bateria de testes textuais

Compara duas entradas textuais sem `id` e sem subgenero/categoria:

1. `title + abstract`;
2. `abstract only`.

Isso testa se o titulo ajuda ou se o modelo textual esta decorando ruido.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC


def treinar_bow(nome, train_text, val_text, test_text):
    rows = []
    best = None
    for C in C_GRID:
        for class_weight in CLASS_WEIGHT_GRID:
            modelo = make_pipeline(
                CountVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2), max_features=50000),
                LinearSVC(C=C, class_weight=class_weight, max_iter=10000),
            )
            modelo.fit(train_text, y_tr)
            pred_tr = modelo.predict(train_text)
            pred_val = modelo.predict(val_text)
            row = {
                'modelo': nome,
                'C': C,
                'class_weight': str(class_weight),
                'f1_macro_treino': f1_score(y_tr, pred_tr, average='macro'),
                'f1_macro_validacao': f1_score(y_val, pred_val, average='macro'),
                'balanced_acc_validacao': balanced_accuracy_score(y_val, pred_val),
            }
            rows.append(row)
            if best is None or row['f1_macro_validacao'] > best[0]['f1_macro_validacao']:
                best = (row, modelo)
    tabela = pd.DataFrame(rows).sort_values('f1_macro_validacao', ascending=False)
    best_row, best_model = best
    pred_test = best_model.predict(test_text)
    resumo = metricas_resumo(nome, y_te, pred_test)
    print('\n###', nome)
    print('Melhor configuracao:', best_row)
    print(pd.Series(resumo).to_string())
    return best_model, pred_test, resumo, tabela

clf_bow_title_abs, y_pred_bow_title_abs, res_bow_title_abs, tabela_bow_title_abs = treinar_bow(
    'BoW_title_abstract', txt_tr, txt_val, txt_te
)
clf_bow_abs, y_pred_bow_abs, res_bow_abs, tabela_bow_abs = treinar_bow(
    'BoW_abstract_only', abs_tr, abs_val, abs_te
)

SAIDA_BOW_GRID = os.path.join(PIPE, 'busca_c_bow_atualizada.csv')
pd.concat([tabela_bow_title_abs, tabela_bow_abs], ignore_index=True).to_csv(SAIDA_BOW_GRID, index=False)
print('\nBusca BoW salva em:', SAIDA_BOW_GRID)

## 13. Inspecao dos pesos do BoW

Aqui os pesos sao interpretaveis: mostram quais termos empurram o SVM para cada categoria no baseline textual.

In [ ]:
vec = clf_bow_title_abs.named_steps['countvectorizer']
svm_bow = clf_bow_title_abs.named_steps['linearsvc']
features = np.array(vec.get_feature_names_out())

for classe in ['cs.LG', 'cs.AI', 'cs.CV', 'hep-ph']:
    if classe not in svm_bow.classes_:
        continue
    i = list(svm_bow.classes_).index(classe)
    pesos = svm_bow.coef_[i]
    top_pos = np.argsort(pesos)[::-1][:15]
    top_neg = np.argsort(pesos)[:10]
    print(f'\nClasse {classe}')
    print('Termos que mais empurram PARA a classe:')
    print(list(zip(features[top_pos], np.round(pesos[top_pos], 3))))
    print('Termos que mais afastam da classe:')
    print(list(zip(features[top_neg], np.round(pesos[top_neg], 3))))

## 14. Comparacao final

In [ ]:
resumo_final = pd.DataFrame([
    res_specter,
    res_bow_title_abs,
    res_bow_abs,
]).round(4)

print(resumo_final.to_string(index=False))

SAIDA_CSV = os.path.join(PIPE, 'comparacao_modelos_metricas_atualizada.csv')
resumo_final.to_csv(SAIDA_CSV, index=False)
print('\nTabela salva em:', SAIDA_CSV)